# 评价表清洗第一步：粒度、候选主键与一单多评

本次先读取原始字符串并检查关系，保留原始记录。评分、评论缺失、日期转换和订单级评价规则放到后续步骤。

一行代表与某个订单关联的评价记录。review_id 和 order_id 都要通过数据检查后，才能判断是否能单独作为主键。

| 字段 | 含义 |
| --- | --- |
| review_id | 评价标识，需检查唯一性 |
| order_id | 该评价记录对应的订单 |
| review_score | 客户体验评分，数据字典定义为 1～5 分 |
| review_comment_title | 客户评论标题 |
| review_comment_message | 客户评论正文 |
| review_creation_date | 评价问卷创建、发送的日期 |
| review_answer_timestamp | 客户提交评价的时间 |

字段说明参照原始目录中的中文数据字典。review_answer_timestamp 不是客服回复时间；review_creation_date 也不能直接当作客户提交评价的时间。没有标题或正文，仍可能有有效评分。

In [7]:
from pathlib import Path
import pandas as pd
from IPython.display import display

project_dir = Path(r'D:\数据分析\olist-operations-analytics')
data_dir = project_dir.parent / 'archive'

reviews_raw = pd.read_csv(
    data_dir / 'olist_order_reviews_dataset.csv',
    dtype=str,
    encoding='utf-8-sig',
    keep_default_na=False,
    na_values=['']
)

print('评价表行数、列数：', reviews_raw.shape)
reviews_raw.head(10)

评价表行数、列数： (99224, 7)


,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18 00:00:00,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10 00:00:00,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17 00:00:00,2018-02-18 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,NaN,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00,2017-04-21 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,NaN,Parabéns lojas lannister adorei comprar pela I...,2018-03-01 00:00:00,2018-03-02 10:26:53
5,15197aa66ff4d0650b5434f1b46cda19,b18dcdf73be66366873cd26c5724d1dc,1,NaN,NaN,2018-04-13 00:00:00,2018-04-16 00:39:37
6,07f9bee5d1b850860defd761afa7ff16,e48aa0d2dcec3a2e87348811bcfdf22b,5,NaN,NaN,2017-07-16 00:00:00,2017-07-18 19:30:34
7,7c6400515c67679fbee952a7525281ef,c31a859e34e3adac22f376954e19b39d,5,NaN,NaN,2018-08-14 00:00:00,2018-08-14 21:36:06
8,a3f6f7f6f433de0aefbb97da197c554c,9c214ac970e84273583ab523dfafd09b,5,NaN,NaN,2017-05-17 00:00:00,2017-05-18 12:05:37
9,8670d52e15e00043ae7de4c01cc2fe06,b9bf720beb4ab3728760088589c62129,4,recomendo,aparelho eficiente. no site a marca do aparelh...,2018-05-22 00:00:00,2018-05-23 16:45:47


In [2]:
reviews_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99224 entries, 0 to 99223
Data columns (total 7 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   review_id                99224 non-null  object
 1   order_id                 99224 non-null  object
 2   review_score             99224 non-null  object
 3   review_comment_title     11568 non-null  object
 4   review_comment_message   40977 non-null  object
 5   review_creation_date     99224 non-null  object
 6   review_answer_timestamp  99224 non-null  object
dtypes: object(7)
memory usage: 5.3+ MB


## 候选主键检查

分别检查 review_id、order_id，以及 order_id + review_id 的组合。

整行重复数为 0，只表示没有两行全部字段相同；不能据此推断某一列唯一。此时先记录结果，不按单列 ID 删除记录。

In [3]:
review_key_cols = ['order_id', 'review_id']

review_key_check = pd.Series({
    '评价记录行数': len(reviews_raw),
    '评价ID不同值数': reviews_raw['review_id'].nunique(),
    '涉及订单数': reviews_raw['order_id'].nunique(),
    '评价ID缺失行数': reviews_raw['review_id'].isna().sum(),
    '订单ID缺失行数': reviews_raw['order_id'].isna().sum(),
    '评价ID重复行数_首条之后': reviews_raw.duplicated(['review_id']).sum(),
    '订单ID重复行数_首条之后': reviews_raw.duplicated(['order_id']).sum(),
    '组合主键重复行数_首条之后': reviews_raw.duplicated(review_key_cols).sum(),
    '整行重复数_首条之后': reviews_raw.duplicated().sum()
})

review_key_check

评价记录行数           99224
评价ID不同值数         98410
涉及订单数            98673
评价ID缺失行数             0
订单ID缺失行数             0
评价ID重复行数_首条之后      814
订单ID重复行数_首条之后      551
组合主键重复行数_首条之后        0
整行重复数_首条之后           0
dtype: int64

## 观察重复的 review_id

找到一个出现多次的 review_id，查看它对应的 order_id。重复评价标识不等于整行重复；先观察关联关系，具体业务原因仍待核查。

In [4]:
repeated_review_ids = reviews_raw.loc[
    reviews_raw['review_id'].duplicated(keep=False),
    'review_id'
]

example_review_id = None
if not repeated_review_ids.empty:
    example_review_id = repeated_review_ids.iloc[0]
    display(reviews_raw.loc[
        reviews_raw['review_id'].eq(example_review_id),
        ['review_id', 'order_id', 'review_score',
         'review_creation_date', 'review_answer_timestamp']
    ])

,review_id,order_id,review_score,review_creation_date,review_answer_timestamp
200,28642ce6250b94cc72bc85960aec6c62,e239d280236cdd3c40cb2c033f681d1c,5,2018-03-25 00:00:00,2018-03-25 21:03:02
58385,28642ce6250b94cc72bc85960aec6c62,bc42a955f289870d5789e6e437206300,5,2018-03-25 00:00:00,2018-03-25 21:03:02


## 检查每笔订单有多少条评价

这里统计评价记录数量，不把重复订单直接视为需要去重。

多条评价的订单数，与首条之后多出来的评价行数是两个指标。例如，某订单有 3 条记录，它是 1 笔多评订单，但额外贡献 2 条记录。

In [5]:
reviews_per_order = reviews_raw['order_id'].value_counts()

print('只有一条评价记录的订单数：', reviews_per_order.eq(1).sum())
print('有多条评价记录的订单数：', reviews_per_order.gt(1).sum())
print('单笔订单最多评价记录数：', reviews_per_order.max())

review_count_distribution = (
    reviews_per_order.value_counts()
    .sort_index()
    .rename_axis('每单评价记录数')
    .reset_index(name='订单数')
)
review_count_distribution

只有一条评价记录的订单数： 98126
有多条评价记录的订单数： 547
单笔订单最多评价记录数： 3


,每单评价记录数,订单数
0,1,98126
1,2,543
2,3,4


In [6]:
example_multi_order_id = None
if reviews_per_order.gt(1).any():
    example_multi_order_id = reviews_per_order.idxmax()
    display(reviews_raw.loc[
        reviews_raw['order_id'].eq(example_multi_order_id),
        ['review_id', 'order_id', 'review_score',
         'review_creation_date', 'review_answer_timestamp']
    ])

,review_id,order_id,review_score,review_creation_date,review_answer_timestamp
1985,ffb8cff872a625632ac983eb1f88843c,c88b1d1b157a9999ce368f218a407141,3,2017-07-22 00:00:00,2017-07-26 13:41:07
82525,202b5f44d09cd3cfc0d6bd12f01b044c,c88b1d1b157a9999ce368f218a407141,5,2017-07-22 00:00:00,2017-07-26 13:40:22
89360,fb96ea2ef8cce1c888f4d45c8e22b793,c88b1d1b157a9999ce368f218a407141,5,2017-07-21 00:00:00,2017-07-26 13:45:15


## 我的理解

1. 评价表有多少行，涉及多少个订单？

我的回答：评价有99224条，涉及98673条订单。

2. review_id 和 order_id 分别能否单独作为本表主键？各用一个检查结果解释。

我的回答：不能，因为review_id和order_id都不是唯一的。

3. order_id + review_id 组合是否非空且唯一？本次是否需要去重？

我的回答：是非空且唯一的，不需要去重。

4. 如果直接把评价明细左连接到订单主表，结果还能保证一行一个订单吗？为什么？

我的回答：不能，因为一个订单可能有多个review_id。

对照基线：99,224 条评价记录、98,673 个订单；review_id 首条之后重复 814 行、order_id 首条之后重复 551 行；组合主键与整行重复均为 0。547 笔订单有多条评价，单笔最多 3 条。

评分与时间检查完成后，我们再确定一单多评的订单级取值规则。当前先保留全部关系，不把任意一条评价直接当作最终订单评价。

In [8]:
reviews_clean = reviews_raw.copy()

score_num = pd.to_numeric(
    reviews_raw['review_score'],
    errors='coerce'
)

score_check = pd.Series({
    '原始缺失数': reviews_raw['review_score'].isna().sum(),
    '非空但解析失败数': (
        reviews_raw['review_score'].notna()
        & score_num.isna()
    ).sum(),
    '非整数评分数': (
        score_num.notna()
        & score_num.mod(1).ne(0)
    ).sum(),
    '超出1至5范围数': (
        score_num.notna()
        & ~score_num.between(1, 5)
    ).sum()
})

score_check

原始缺失数       0
非空但解析失败数    0
非整数评分数      0
超出1至5范围数    0
dtype: int64

In [9]:
if not score_num.isin([1, 2, 3, 4, 5]).all():
    raise ValueError('存在无效评分，请先核查')

reviews_clean['review_score'] = score_num.astype('Int64')

reviews_clean['is_low_score'] = (
    reviews_clean['review_score'].le(2)
)

print('低评分评价记录数：', reviews_clean['is_low_score'].sum())

reviews_clean['review_score'].value_counts().sort_index()

低评分评价记录数： 14575


review_score
1    11424
2     3151
3     8179
4    19142
5    57328
Name: count, dtype: Int64

In [10]:
text_flags = {
    'review_comment_title': 'has_comment_title',
    'review_comment_message': 'has_comment_message'
}

text_check = {}

for col, flag_col in text_flags.items():
    original_text = reviews_raw[col]

    has_text = (
        original_text.fillna('')
        .str.strip()
        .ne('')
    )

    reviews_clean[flag_col] = has_text

    text_check[col] = {
        '原始缺失数': original_text.isna().sum(),
        '非空但只有空白字符数': (
            original_text.notna() & ~has_text
        ).sum(),
        '无有效文字记录数': (~has_text).sum(),
        '有有效文字记录数': has_text.sum()
    }

pd.DataFrame(text_check).T

,原始缺失数,非空但只有空白字符数,无有效文字记录数,有有效文字记录数
review_comment_title,87656,2,87658,11566
review_comment_message,58247,27,58274,40950


## 时间

In [11]:
review_time_cols = [
    'review_creation_date',
    'review_answer_timestamp'
]

review_time_check = {}

for col in review_time_cols:
    parsed = pd.to_datetime(
        reviews_raw[col],
        format='%Y-%m-%d %H:%M:%S',
        errors='coerce'
    )

    review_time_check[col] = {
        '原始缺失数': reviews_raw[col].isna().sum(),
        '非空但解析失败数': (
            reviews_raw[col].notna() & parsed.isna()
        ).sum(),
        '转换后类型': str(parsed.dtype)
    }

    reviews_clean[col] = parsed

pd.DataFrame(review_time_check).T

,原始缺失数,非空但解析失败数,转换后类型
review_creation_date,0,0,datetime64[ns]
review_answer_timestamp,0,0,datetime64[ns]


In [12]:
reviews_clean['can_compare_review_time'] = (
    reviews_clean[review_time_cols]
    .notna()
    .all(axis=1)
)

reviews_clean['flag_answer_before_creation'] = (
    reviews_clean['can_compare_review_time']
    & reviews_clean['review_answer_timestamp'].lt(
        reviews_clean['review_creation_date']
    )
)

pd.Series({
    '可比较评价记录数': (
        reviews_clean['can_compare_review_time'].sum()
    ),
    '无法比较评价记录数': (
        ~reviews_clean['can_compare_review_time']
    ).sum(),
    '提交早于问卷创建记录数': (
        reviews_clean['flag_answer_before_creation'].sum()
    )
})

可比较评价记录数       99224
无法比较评价记录数          0
提交早于问卷创建记录数        0
dtype: int64

In [13]:
valid_review_time = (
    reviews_clean['can_compare_review_time']
    & ~reviews_clean['flag_answer_before_creation']
)

reviews_clean['review_answer_lag_days'] = (
    (
        reviews_clean['review_answer_timestamp']
        - reviews_clean['review_creation_date']
    )
    .dt.total_seconds()
    .div(86400)
    .where(valid_review_time)
)

reviews_clean['review_answer_lag_days'].describe(
    percentiles=[0.5, 0.9, 0.99]
).round(2)

count    99224.00
mean         3.15
std          9.89
min          0.09
50%          1.67
90%          4.86
99%         22.07
max        518.70
Name: review_answer_lag_days, dtype: float64

## 把评价记录整理成一行一个订单

In [ ]:
valid_reviews = reviews_clean.loc[
    reviews_clean['review_score'].isin([1, 2, 3, 4, 5])
    & reviews_clean['review_answer_timestamp'].notna()
].copy()

#筛选每笔订单最新的一条评价
latest_time = valid_reviews.groupby('order_id')[
    'review_answer_timestamp'
].transform('max')

latest_reviews = valid_reviews.loc[
    valid_reviews['review_answer_timestamp'].eq(latest_time)
].copy()

tie_order_count = (
    latest_reviews['order_id']
    .value_counts()
    .gt(1)
    .sum()
)

print('最新提交时间存在并列的订单数：', tie_order_count)

if tie_order_count > 0:
    raise ValueError('存在并列最新评价，需要先制定取值规则')

最新提交时间存在并列的订单数： 0


In [ ]:
#把全部评价聚合成订单级统计
order_review_stats = valid_reviews.groupby(
    'order_id', as_index=False
).agg(
    review_record_count=('review_id', 'size'),
    any_low_score=('is_low_score', 'any')
)

latest_result = latest_reviews[
    [
        'order_id', 'review_id', 'review_score',
        'review_answer_timestamp', 'is_low_score'
    ]
].rename(columns={
    'review_id': 'latest_review_id',
    'review_score': 'latest_review_score',
    'review_answer_timestamp': 'latest_review_answer_timestamp',
    'is_low_score': 'latest_is_low_score'
})

order_reviews = order_review_stats.merge(
    latest_result,
    on='order_id',
    how='left',
    validate='one_to_one'
)

order_reviews['rules_disagree'] = (
    order_reviews['any_low_score']
    .ne(order_reviews['latest_is_low_score'])
)
#False有低分，最新就是低分，一致；True有低分，但最新评价是高分，矛盾；True原本无低分，最新评价是低分，矛盾；False全程无低分，一致
print('订单级评价表行数：', len(order_reviews))
order_reviews.head()

订单级评价表行数： 98673


,order_id,review_record_count,any_low_score,latest_review_id,latest_review_score,latest_review_answer_timestamp,latest_is_low_score,rules_disagree
0,00010242fe8c5a6d1ba2dd792cb16214,1,False,97ca439bc427b48bc1cd7177abe71365,5,2017-09-22 10:57:03,False,False
1,00018f77f2f0320c557190d7a144bdd3,1,False,7b07bacd811c4117b742569b04ce3580,4,2017-05-15 11:34:13,False,False
2,000229ec398224ef6ca0657da4fc703e,1,False,0c5b33dea94867d1ac402749e5438e8b,5,2018-01-23 16:06:31,False,False
3,00024acbcdf0a6daa1e931b038114c75,1,False,f4028d019cb58564807486a6aaf33817,4,2018-08-15 16:39:01,False,False
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,False,940144190dcba6351888cafa43f3a3a5,5,2017-03-03 10:54:59,False,False


In [16]:
rule_comparison = pd.DataFrame({
    '低评分订单数': [
        order_reviews['latest_is_low_score'].sum(),
        order_reviews['any_low_score'].sum()
    ],
    '有有效评价订单数': len(order_reviews)
}, index=['最新有效评分', '任意一次低评分'])

rule_comparison['低评分比例（%）'] = (
    rule_comparison['低评分订单数']
    / rule_comparison['有有效评价订单数']
    * 100
).round(2)

print('两种规则判定不同的订单数：',
      order_reviews['rules_disagree'].sum())

rule_comparison

两种规则判定不同的订单数： 39


,低评分订单数,有有效评价订单数,低评分比例（%）
最新有效评分,14494,98673,14.69
任意一次低评分,14533,98673,14.73


## 规则差异样本检查和评价覆盖检查

In [ ]:
example_order_id = order_reviews.loc[
    order_reviews['rules_disagree'], 'order_id' #评价逻辑矛盾的订单
].sort_values().iloc[0]

reviews_clean.loc[
    reviews_clean['order_id'].eq(example_order_id),
    ['order_id', 'review_score', 'review_answer_timestamp']
].sort_values('review_answer_timestamp')

,order_id,review_score,review_answer_timestamp
15775,059bbeb3477ed66fd7e670c3f879009a,2,2017-08-01 20:39:04
9139,059bbeb3477ed66fd7e670c3f879009a,5,2017-08-04 09:32:59


In [18]:
orders_base = pd.read_csv(
    data_dir / 'olist_orders_dataset.csv',
    dtype=str,
    encoding='utf-8-sig',
    keep_default_na=False,
    na_values=['']
)

# 先反向检查，避免左连接遗漏无法匹配主表的评价订单
orphan_count = (
    ~order_reviews['order_id'].isin(orders_base['order_id'])
).sum()

print('评价汇总中找不到主表的订单数：', orphan_count)

if orphan_count > 0:
    raise ValueError('存在无法匹配订单主表的评价订单，请先核查')

review_summary = order_reviews.copy()

# 在关联前使用可空类型，保留未匹配订单的未知状态
for col in ['review_record_count', 'latest_review_score']:
    review_summary[col] = review_summary[col].astype('Int64')

for col in ['any_low_score', 'latest_is_low_score', 'rules_disagree']:
    review_summary[col] = review_summary[col].astype('boolean')

orders_with_reviews = orders_base.merge(
    review_summary,
    on='order_id',
    how='left',
    validate='one_to_one',
    indicator='reviews_match'
)

orders_with_reviews['flag_missing_reviews'] = (
    orders_with_reviews['reviews_match'].eq('left_only')
)

print('关联后订单行数：', len(orders_with_reviews))
orders_with_reviews['reviews_match'].value_counts()

评价汇总中找不到主表的订单数： 0
关联后订单行数： 99441


reviews_match
both          98673
left_only       768
right_only        0
Name: count, dtype: int64

In [19]:
missing_review_orders = orders_with_reviews.loc[
    orders_with_reviews['flag_missing_reviews']
].copy()

display(missing_review_orders['order_status'].value_counts())

missing_review_orders[
    [
        'order_id',
        'order_status',
        'latest_review_score',
        'latest_is_low_score',
        'any_low_score'
    ]
].head()

order_status
delivered      646
shipped         75
canceled        20
unavailable     14
processing       6
invoiced         5
created          2
Name: count, dtype: int64

,order_id,order_status,latest_review_score,latest_is_low_score,any_low_score
16,403b97836b0c04a622354cf531062e5f,delivered,<NA>,<NA>,<NA>
154,6942b8da583c2f9957e990d028607019,shipped,<NA>,<NA>,<NA>
311,4906eeadde5f70b308c20c4a8f20be02,delivered,<NA>,<NA>,<NA>
382,b7a4a9ecb1cd3ef6a3e36a48e200e3be,delivered,<NA>,<NA>,<NA>
390,59b32faedc12322c672e95ec3716d614,delivered,<NA>,<NA>,<NA>
